
# TP : Modélisation de l'achalandage cycliste à Montréal

Ce travail pratique vise à modéliser l'achalandage cycliste en fonction :
- du trafic motorisé
- du type de voie cyclable
- de la longueur de la piste
- de la présence de travaux
- de la saison

Nous utiliserons des données ouvertes de la Ville de Montréal pour construire un modèle supervisé.


### Ce modèle peut aider à répondre à des questions comme :

- Quels types de pistes attirent le plus de cyclistes dans des zones à fort trafic ?
- Les travaux routiers réduisent-ils l’achalandage cycliste ?
- Est-ce que les pistes dans des zones calmes sont plus utilisées ?
- Il est aussi possible d'ajouter la meteo pour prédire l'acalendage selon la météo (API)


## Liens de téléchargement des données

- [Comptage des vélos sur les pistes cyclables](https://donnees.montreal.ca/dataset/velos-comptage)
- [Comptage des véhicules, cyclistes et piétons aux intersections](https://donnees.montreal.ca/dataset/comptage-vehicules-pietons)
- [Travaux routiers en cours (Info-travaux)](https://donnees.montreal.ca/en/dataset/info-travaux)
- [Réseau cyclable](https://donnees.montreal.ca/fr/dataset/pistes-cyclables)
- [API météo](https://archive-api.open-meteo.com/v1/archive)


## Description des jeux de données
- La fusion de multiples jeux de données hétérogènes
- Un modèle supervisé simple mais parlant
- **Comptage cycliste** : nombre de passages de vélos par jour et par site de comptage.
- **Comptage routier** : nombre de véhicules, cyclistes et piétons par intersection et par période.
- **Travaux routiers** : informations sur les chantiers en cours, leur durée et leur localisation.
- **Réseau cyclable** : géométrie et attributs des pistes cyclables (type, longueur, accessibilité).
- **API météo** : les données météo de montréal latitude=45.5017&longitude=-73.5673&timezone=America/Toronto

In [ ]:

import pandas as pd
import json
import numpy as np
from sklearn.neighbors import BallTree

# ============================================================
# 1. CHARGEMENT DES DONNÉES
# ============================================================

# Charger les fichiers CSV
df_velo = pd.read_csv("C:/Users/Public/Documents/TELECHARGEMENT/BdeB/A52-Algorithme d'apprentissage supervisés/TP_Modele/comptage_velo_2025.csv")
df_comptages = pd.read_csv("C:/Users/Public/Documents/TELECHARGEMENT/BdeB/A52-Algorithme d'apprentissage supervisés/TP_Modele/comptages_vehicules_cyclistes_pietons.csv")
df_entraves = pd.read_csv("C:/Users/Public/Documents/TELECHARGEMENT/BdeB/A52-Algorithme d'apprentissage supervisés/TP_Modele/entraves-travaux-en-cours.csv")

# Charger le réseau cyclable (JSON → DataFrame)
with open("C:/Users/Public/Documents/TELECHARGEMENT/BdeB/A52-Algorithme d'apprentissage supervisés/TP_Modele/reseau_cyclable.json", 
          "r", 
          encoding="utf-8"
          ) as f:
    data_json = json.load(f)
df_reseau = pd.json_normalize(data_json["features"])

print("Données chargées avec succès !")
print("Taille df_velo :", df_velo.shape)
print("Taille df_comptages :", df_comptages.shape)
print("Taille df_entraves :", df_entraves.shape)
print("Taille df_reseau :", df_reseau.shape)

# ============================================================
# 2. JOINTURE VELO : COMPTAGES (par proximité géographique)
# ============================================================

# Extraire coordonnées (latitude, longitude) en radians
velo_coords = np.radians(df_velo[["latitude", "longitude"]].values)
comp_coords = np.radians(df_comptages[["Latitude", "Longitude"]].dropna().values)

# Construire un BallTree (recherche de voisin le plus proche avec Haversine)
tree_comp = BallTree(comp_coords, metric="haversine")

# Trouver l'intersection la plus proche pour CHAQUE compteur vélo
distances, indices = tree_comp.query(velo_coords, k=1)

# Ajouter colonnes au DataFrame vélo
df_velo["closest_intersection_id"] = df_comptages.iloc[indices.flatten()]["Id_Reference"].values
df_velo["closest_intersection_name"] = df_comptages.iloc[indices.flatten()]["Nom_Intersection"].values
df_velo["distance_intersection_km"] = distances.flatten() * 6371  # conversion radian → km

print("\nJointure vélo: intersections terminée")

# ============================================================
# 3. AJOUT DES ENTRAVES (travaux proches des compteurs vélo)
# ============================================================

# Construire un BallTree pour les entraves
entraves_coords = np.radians(df_entraves[["latitude", "longitude"]].dropna().values)
tree_ent = BallTree(entraves_coords, metric="haversine")

# Trouver le chantier le plus proche pour chaque compteur vélo
dist_ent, idx_ent = tree_ent.query(velo_coords, k=1)

# Ajouter colonnes
df_velo["closest_entrave_id"] = df_entraves.iloc[idx_ent.flatten()]["id"].values
df_velo["closest_entrave_distance_km"] = dist_ent.flatten() * 6371

print("Jointure vélo: entraves terminée")

# ============================================================
# 4. AJOUT DU RÉSEAU CYCLABLE (proximité)
# ============================================================

# Certains segments du réseau cyclable ont des coordonnées complexes (LineString)
# Ici, on simplifie : on prend seulement le premier point de chaque ligne
df_reseau["latitude"] = df_reseau["geometry.coordinates"].apply(lambda x: x[0][1])
df_reseau["longitude"] = df_reseau["geometry.coordinates"].apply(lambda x: x[0][0])

# Construire un BallTree pour le réseau cyclable
reseau_coords = np.radians(df_reseau[["latitude", "longitude"]].dropna().values)
tree_res = BallTree(reseau_coords, metric="haversine")

# Trouver la piste la plus proche pour chaque compteur vélo
dist_res, idx_res = tree_res.query(velo_coords, k=1)

# Ajouter colonnes
df_velo["closest_piste_id"] = df_reseau.iloc[idx_res.flatten()]["properties.ID_CYCL"].values
df_velo["closest_piste_distance_km"] = dist_res.flatten() * 6371

print("Jointure vélo: réseau cyclable terminée")
print('----------------------------------------------------------------')


# ============================================================
# 5. DATASET AVANT MÉTÉO
# ============================================================
df_velo.head()
print('----------------------------------------------------------------')
df_velo.info()
print('----------------------------------------------------------------')


# ============================================================
# 6. AJOUT DU DATASET MÉTÉO
# ============================================================
print('----------------------------------------------------------------')
# 1. Charger le fichier météo (le tien ressemble à ce format CSV)
df_meteo = pd.read_csv("C:/Users/Public/Documents/TELECHARGEMENT/BdeB/A52-Algorithme d'apprentissage supervisés/TP_Modele/meteo_montreal_horaire.csv")

# Vérifier que les colonnes sont bien lues
print("Colonnes du dataset météo :", df_meteo.columns.tolist())
display(df_meteo.head())  # aperçu
print('----------------------------------------------------------------')
# 2. Vérifier le format de la date et de l'heure
# Dans ton CSV : date = "2022-01-01", heure = "00:00:00"
# On garde tel quel pour que ça corresponde à df_velo

# 3. Nettoyer les noms de colonnes (exemple : "humiditÃ©_pct" -> "humidite_pct")
df_meteo = df_meteo.rename(columns={
    "humiditÃ©_pct": "humidite_pct"
})

# 4. Vérifier les colonnes météo
# Doivent être : date, heure, temp_c, precip_mm, wind_kmh, humidite_pct, cloud_pct
print("Colonnes météo après renommage :", df_meteo.columns.tolist())
print('----------------------------------------------------------------')

# 5. Fusion météo avec df_final (celui qu’on a déjà obtenu avec les autres jointures)
# On utilise "date" + "heure" comme clés de jointure
df_final = df_velo.merge(
    df_meteo,
    on=["date", "heure"],  # jointure sur date + heure
    how="left"             # garder toutes les lignes vélo, même si météo manquante
)
print("Jointure vélo: donnée météo terminées")
# 6. Vérifier le résultat
print("Données finales avec météo intégrée :")
print(df_final.head(10))
print('----------------------------------------------------------------')


# ============================================================
# . DATASET FINAL
# ============================================================
print("\nAperçu du DataFrame final :")

display(df_final.head())
print('----------------------------------------------------------------')
df_final.info()
print('----------------------------------------------------------------')
# Sauvegarder en CSV pour l'EDA
df_final.to_csv("C:/Users/Public/Documents/TELECHARGEMENT/BdeB/A52-Algorithme d'apprentissage supervisés/TP_Modele/dataset_fusionne.csv", index=False)
print("\n Dataset fusionné sauvegardé dans dataset_fusionne.csv")


Données chargées avec succès !
Taille df_velo : (775481, 6)
Taille df_comptages : (273250, 30)
Taille df_entraves : (2363, 43)
Taille df_reseau : (9314, 26)

Jointure vélo: intersections terminée
Jointure vélo: entraves terminée
Jointure vélo: réseau cyclable terminée
Colonnes du dataset météo : ['date', 'heure', 'temp_c', 'precip_mm', 'wind_kmh', 'humidité_pct', 'cloud_pct']
         date     heure  temp_c  precip_mm  wind_kmh  humidité_pct  cloud_pct
0  2022-01-01  00:00:00    -2.8        0.0       2.1            99        100
1  2022-01-01  01:00:00    -3.7        0.0       2.3            99        100
2  2022-01-01  02:00:00    -2.9        0.0       4.5            98        100
Colonnes météo après renommage : ['date', 'heure', 'temp_c', 'precip_mm', 'wind_kmh', 'humidité_pct', 'cloud_pct']
Jointure vélo: donnée météo terminées
Données finales avec météo intégrée :
         date     heure  id_compteur  nb_passages  longitude   latitude  \
0  2025-01-01  00:00:00    100054073       

,date,heure,id_compteur,nb_passages,longitude,latitude,closest_intersection_id,closest_intersection_name,distance_intersection_km,closest_entrave_id,closest_entrave_distance_km,closest_piste_id,closest_piste_distance_km,temp_c,precip_mm,wind_kmh,humidité_pct,cloud_pct
0,2025-01-01,00:00:00,100054073,0.0,-73.590636,45.560713,10476,16 e Avenue / Bélanger,0.027686,68c825a2a9e0000012adad78,0.148111,24819,0.027747,0.9,0.0,10.4,88.0,98.0
1,2025-01-01,00:00:00,300021685,0.0,-73.613370,45.631590,10361,Louis-Hippolyte-La Fontaine / Perras Int.Est,0.249476,68c3121da9e0000012ad5c7f,0.381940,27217,0.019238,0.9,0.0,10.4,88.0,98.0
2,2025-01-01,00:00:00,100003040,0.0,-73.544410,45.501270,10999,boulevard Saint-Laurent / rue de la Commune,0.821999,67bc87afaa353d001234df00,0.255474,28542,0.043867,0.9,0.0,10.4,88.0,98.0
3,2025-01-01,00:00:00,100052606,0.0,-73.538818,45.555084,10994,Rouen / Saint-Clément,0.510626,68b9c893640f320012846919,0.289102,22976,0.048889,0.9,0.0,10.4,88.0,98.0
4,2025-01-01,00:00:00,100003032,2.0,-73.562970,45.516216,10525,Berri / Sainte-Catherine,0.311038,68c1c5848f7d590012d5b9af,0.090427,25017,0.015040,0.9,0.0,10.4,88.0,98.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 775481 entries, 0 to 775480
Data columns (total 18 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   date                         775481 non-null  object 
 1   heure                        775481 non-null  object 
 2   id_compteur                  775481 non-null  int64  
 3   nb_passages                  775477 non-null  float64
 4   longitude                    775481 non-null  float64
 5   latitude                     775481 non-null  float64
 6   closest_intersection_id      775481 non-null  int64  
 7   closest_intersection_name    775481 non-null  object 
 8   distance_intersection_km     775481 non-null  float64
 9   closest_entrave_id           775481 non-null  object 
 10  closest_entrave_distance_km  775481 non-null  float64
 11  closest_piste_id             775481 non-null  int64  
 12  closest_piste_distance_km    775481 non-null  float64
 13 